In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
filename = "G:/neural_network_from_scratch/bank marketing/bank-additional-full.csv"
df = pd.read_csv(filename, sep=";")
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [4]:
df_2 = df.drop(columns=['age','campaign','marital','education','month','day_of_week','housing', 'loan','duration'],axis=1)

In [5]:
df_2.columns

Index(['job', 'default', 'contact', 'pdays', 'previous', 'poutcome',
       'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m',
       'nr.employed', 'y'],
      dtype='object')

In [6]:
df_2['previously_contacted'] = (df_2['pdays'] != 999).astype(int)
df_2['pdays_actual'] = df_2['pdays'].replace(999, np.nan)
df_2['pdays_actual'] = df_2['pdays_actual'].fillna(0)
df_2.drop(columns=['pdays'], inplace=True)

In [7]:
categorical_features = [
    "job",
    "default",
    "contact",
    "poutcome",
    "y"
]

df_enc = pd.get_dummies(df_2,columns=categorical_features,dtype=int)
df_enc.head()


,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,previously_contacted,pdays_actual,job_admin.,job_blue-collar,...,default_no,default_unknown,default_yes,contact_cellular,contact_telephone,poutcome_failure,poutcome_nonexistent,poutcome_success,y_no,y_yes
0,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,0,0,...,1,0,0,0,1,0,1,0,1,0
1,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,0,0,...,0,1,0,0,1,0,1,0,1,0
2,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,0,0,...,1,0,0,0,1,0,1,0,1,0
3,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,1,0,...,1,0,0,0,1,0,1,0,1,0
4,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,0,0,...,1,0,0,0,1,0,1,0,1,0


In [8]:
X = df_enc.copy().drop(columns=["y_no","y_yes"],axis=1)
y = df_enc['y_yes'].values.reshape(-1,1)

print(X.shape)
print(y.shape)

(41188, 28)
(41188, 1)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

continuous_cols = ['previous','pdays_actual','emp.var.rate','cons.price.idx','cons.conf.idx', 'euribor3m', 'nr.employed']

mean = X_train[continuous_cols].mean()
std = X_train[continuous_cols].std()

X_train[continuous_cols] = (
    X_train[continuous_cols] - mean
) / std

X_val[continuous_cols] = (
    X_val[continuous_cols] - mean
) / std

In [10]:
X_train.shape

(32950, 28)

In [11]:
class LayerDense:
    def __init__(self,inputs,neurons):
        self.weights = np.random.randn(inputs,neurons) + np.sqrt(1/inputs)
        self.biases = np.zeros((1,neurons))
    def forward(self,X):
        self.output = np.dot(X,self.weights) + self.biases
        return self.output
    
def Relu(z):
    return np.maximum(0,z)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1/(1+np.exp(-z))

def loss_func(y_pred,y):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1-epsilon)
    loss = -np.mean(y*np.log(y_pred)+(1-y)*np.log(1-y_pred))
    return loss

In [12]:
layer_1 = LayerDense(28,64)
layer_2 = LayerDense(64,32)
layer_3 = LayerDense(32,1)

In [13]:
def update_parameters(dw1,db1,dw2,db2,dw3,db3,lr):
    
    layer_1.weights -= lr * dw1
    layer_1.biases -= lr * db1
    layer_2.weights -= lr * dw2
    layer_2.biases -= lr * db2
    layer_3.weights -= lr * dw3
    layer_3.biases -= lr * db3

In [14]:
X_train = np.asarray(X_train, dtype=np.float64)
y_train = np.asarray(y_train, dtype=np.float64)

In [15]:
best_loss = float('inf')

best_W1 = None
best_b1 = None
best_W2 = None
best_b2 = None
best_W3 = None
best_b3 = None

In [16]:
learning_rate = 0.25
epochs = 500
batch_size = 256

loss_history = []

for epoch in range(1, epochs + 1):

    indices = np.random.permutation(X_train.shape[0])

    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]

    epoch_loss = 0

    for start in range(0, X_train.shape[0], batch_size):

        end = start + batch_size

        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        m = X_batch.shape[0]

        z1 = layer_1.forward(X_batch)
        a1 = Relu(z1)

        z2 = layer_2.forward(a1)
        a2 = Relu(z2)

        z3 = layer_3.forward(a2)
        a3 = sigmoid(z3)

        loss = loss_func(a3, y_batch)

        epoch_loss += loss * m

        dz3 = a3 - y_batch

        dw3 = np.dot(a2.T, dz3) / m
        db3 = np.sum(dz3, axis=0, keepdims=True) / m

        da2 = np.dot(dz3, layer_3.weights.T)
        dz2 = da2 * (z2 > 0)

        dw2 = np.dot(a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        da1 = np.dot(dz2, layer_2.weights.T)
        dz1 = da1 * (z1 > 0)

        dw1 = np.dot(X_batch.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        update_parameters(
            dw1, db1,
            dw2, db2,
            dw3, db3,
            learning_rate
        )

    epoch_loss /= X_train.shape[0]
    loss_history.append(epoch_loss)

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        best_W1 = layer_1.weights.copy()
        best_b1 = layer_1.biases.copy()

        best_W2 = layer_2.weights.copy()
        best_b2 = layer_2.biases.copy()

        best_W3 = layer_3.weights.copy()
        best_b3 = layer_3.biases.copy()

    if epoch % 50 == 0:
        print(f"Epoch: {epoch} | Loss: {epoch_loss:.6f}")

Epoch: 50 | Loss: 0.280002
Epoch: 100 | Loss: 0.275090
Epoch: 150 | Loss: 0.272238
Epoch: 200 | Loss: 0.271628
Epoch: 250 | Loss: 0.271273
Epoch: 300 | Loss: 0.269783
Epoch: 350 | Loss: 0.269159
Epoch: 400 | Loss: 0.268773
Epoch: 450 | Loss: 0.267794
Epoch: 500 | Loss: 0.267425


In [17]:
layer_1.weights = best_W1.copy()
layer_1.biases = best_b1.copy()

layer_2.weights = best_W2.copy()
layer_2.biases = best_b2.copy()

layer_3.weights = best_W3.copy()
layer_3.biases = best_b3.copy()

In [18]:
# on validation set
z1 = layer_1.forward(X_val)
a1 = Relu(z1)
z2 = layer_2.forward(a1)
a2 = Relu(z2)
z3 = layer_3.forward(a2)
a3 = sigmoid(z3)

y_pred = (a3 >= 0.3).astype(int)
accuracy = np.mean(y_pred == y_val) * 100
print("Validation Accuracy:", accuracy)

Validation Accuracy: 88.54090798737558


In [19]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = (a3 >= 0.3).astype(int)
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

[[6844  466]
 [ 478  450]]
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      7310
           1       0.49      0.48      0.49       928

    accuracy                           0.89      8238
   macro avg       0.71      0.71      0.71      8238
weighted avg       0.88      0.89      0.89      8238



In [21]:
np.savez(
    "model_v1.npz",
    w1=best_W1,
    b1=best_b1,
    w2=best_W2,
    b2=best_b2,
    w3=best_W3,
    b3=best_b3,
    mean=mean.values,
    std=std.values
)
print("Model Saved !")

Model Saved !


In [25]:
X.head(1)

,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,previously_contacted,pdays_actual,job_admin.,job_blue-collar,...,job_unemployed,job_unknown,default_no,default_unknown,default_yes,contact_cellular,contact_telephone,poutcome_failure,poutcome_nonexistent,poutcome_success
0,0,1.1,93.994,-36.4,4.857,5191.0,0,0.0,0,0,...,0,0,1,0,0,0,1,0,1,0
